# Module 04 — Exercises

These exercises mirror the sections of the [concept guide](README.md): numerical derivatives,
activation derivatives, gradients, Jacobians, Hessians, Taylor models, matrix calculus and the
gradients of ML losses, ending with a reusable gradient checker. Use only NumPy (no autodiff
libraries, no `scipy.misc.derivative`).

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def reference_grad_fd(f, x, h=1e-6):
    # Reference central-difference gradient used by the self-checks below.
    # Exercise 1 asks you to write your own version — try it without peeking.
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    for i in range(x.size):
        e = np.zeros_like(x); e[i] = h
        g[i] = (f(x + e) - f(x - e)) / (2 * h)
    return g

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — Central differences in one and many dimensions  `★☆☆`

Implement two numerical-derivative helpers using **central differences**:

1. `central_diff(f, x, h=1e-6)` for a scalar function $f:\mathbb{R}\to\mathbb{R}$, returning
   $\dfrac{f(x+h)-f(x-h)}{2h}$ as a float.
2. `grad_fd(f, x, h=1e-6)` for $f:\mathbb{R}^n\to\mathbb{R}$ taking a 1-D NumPy array `x`,
   returning the gradient as an array of the same shape: perturb one coordinate at a time by
   $\pm h$ (use the standard basis vectors $\mathbf{e}_i$).

Use only NumPy. Do not look at `reference_grad_fd` in the setup cell until your checks pass.

In [ ]:
def central_diff(f, x, h=1e-6):
    # YOUR CODE HERE
    return None

def grad_fd(f, x, h=1e-6):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    got = central_diff(lambda t: t**3, 2.0, 0.1)
except Exception:
    got = None
check_close('central difference of x^3 at 2 with h = 0.1 equals 12.01', got, 12.01, tol=1e-9)

try:
    got = central_diff(np.exp, 1.0)
except Exception:
    got = None
check_close('central difference of exp at 1 is e (to 1e-8)', got, np.e, tol=1e-8)

try:
    got = grad_fd(lambda v: v @ v, np.array([1.0, 2.0, 3.0]))
except Exception:
    got = None
check_close('gradient of x.x at (1,2,3) is (2,4,6)', got, np.array([2.0, 4.0, 6.0]), tol=1e-7)

try:
    got = grad_fd(lambda v: np.sum(np.sin(v)) * v[0], np.array([0.3, -1.2, 2.0, 0.7]))
    exp_ = reference_grad_fd(lambda v: np.sum(np.sin(v)) * v[0], np.array([0.3, -1.2, 2.0, 0.7]))
except Exception:
    got, exp_ = None, None
check_close('gradient of a 4-D function matches the reference', got, exp_, tol=1e-7)
check('grad_fd returns an array with the shape of x', got is not None and np.shape(got) == (4,))

## Exercise 2 — The step-size trade-off  `★☆☆`

Write `fd_errors(f, x, exact, hs)` that returns, for each step size in the 1-D array `hs`, the
absolute error $\left|\dfrac{f(x+h)-f(x-h)}{2h} - \texttt{exact}\right|$ of the central-difference
derivative at the scalar point `x` (vectorise over `hs`; `f` is a NumPy ufunc such as `np.exp`).
Then write `best_step(f, x, exact, hs)` returning the step size (a float) with the smallest error.

Use `hs = np.logspace(-14, -1, 53)` and $f = \exp$ at $x = 1$ (exact derivative $e$) to see the
V-shaped curve from the lesson. Where is the minimum, and how small is the best error?

In [ ]:
def fd_errors(f, x, exact, hs):
    # YOUR CODE HERE
    return None

def best_step(f, x, exact, hs):
    # YOUR CODE HERE
    return None

hs = np.logspace(-14, -1, 53)
# optional: plt.loglog(hs, fd_errors(np.exp, 1.0, np.e, hs)); plt.show()

**Check your answer:**

In [ ]:
hs = np.logspace(-14, -1, 53)
try:
    errs = np.asarray(fd_errors(np.exp, 1.0, np.e, hs), dtype=float)
    if errs.shape != hs.shape:
        errs = None
except Exception:
    errs = None
check('fd_errors returns one error per step size', errs is not None)
check_close('error at h = 0.1 matches the closed form', None if errs is None else errs[-1],
            abs((np.exp(1.1) - np.exp(0.9)) / 0.2 - np.e), tol=1e-12)
try:
    hb = float(best_step(np.exp, 1.0, np.e, hs))
except Exception:
    hb = None
check('best step size lies between 1e-7 and 1e-4 (round-off vs truncation balance)', hb is not None and 1e-7 <= hb <= 1e-4,
      hint='think about where the two error sources are equal')
check('best error is below 1e-8', errs is not None and float(np.min(errs)) < 1e-8)
check('error at h = 1e-13 is worse than at h = 1e-5 (round-off dominates)',
      errs is not None and errs[np.argmin(np.abs(hs - 1e-13))] > errs[np.argmin(np.abs(hs - 1e-5))])

## Exercise 3 — Derivatives of the activation functions  `★☆☆`

Implement the derivatives of the four standard activations as NumPy functions of an array `z`:
`d_sigmoid(z)`, `d_tanh(z)`, `d_softplus(z)` (where $\mathrm{softplus}(z) = \ln(1+e^z)$) and
`d_relu(z)` (return `0.0` at $z = 0$, the usual subgradient choice). Derive each formula with the
chain rule first — you should only need `sigmoid` (defined in the setup cell) and `np.tanh`, not
`np.exp` directly.

In [ ]:
def d_sigmoid(z):
    # YOUR CODE HERE
    return None

def d_tanh(z):
    # YOUR CODE HERE
    return None

def d_softplus(z):
    # YOUR CODE HERE
    return None

def d_relu(z):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
zz = np.array([-3.0, -0.7, 0.2, 1.5, 4.0])
h = 1e-6
pairs = [('sigmoid', d_sigmoid, sigmoid), ('tanh', d_tanh, np.tanh),
         ('softplus', d_softplus, lambda t: np.logaddexp(0.0, t)), ('ReLU', d_relu, lambda t: np.maximum(0.0, t))]
for name, dact, act in pairs:
    try:
        got = dact(zz)
    except Exception:
        got = None
    check_close(f'd_{name} matches a central difference at 5 points', got, (act(zz + h) - act(zz - h)) / (2 * h), tol=1e-6)
try:
    got = d_sigmoid(0.0)
except Exception:
    got = None
check_close("sigma'(0) = 1/4", got, 0.25)
try:
    got = d_relu(np.array([-1.0, 0.0, 2.0]))
except Exception:
    got = None
check_close("ReLU' at (-1, 0, 2) is (0, 0, 1)", got, np.array([0.0, 0.0, 1.0]))

## Exercise 4 — Why the gradient is the direction of steepest ascent  `★☆☆`

A colleague says: *"The gradient points uphill because each component $\partial f/\partial x_i$ is
positive when $f$ increases in $x_i$."* That is true but does not explain why the *combined*
direction is the steepest one. Give the real argument, and state (a) what the maximal slope is,
(b) in which direction the slope is zero, and (c) what this implies about the angle between the
gradient and the level set through the point.

<details><summary><b>Show answer</b></summary>

For a unit vector $\mathbf{u}$ the slope in direction $\mathbf{u}$ is the directional derivative
$D_{\mathbf{u}}f = \nabla f^\top\mathbf{u} = \lVert\nabla f\rVert\,\lVert\mathbf{u}\rVert\cos\phi = \lVert\nabla f\rVert\cos\phi$,
where $\phi$ is the angle between $\mathbf{u}$ and $\nabla f$. By Cauchy–Schwarz this is at most
$\lVert\nabla f\rVert$, with equality exactly when $\phi = 0$, i.e. $\mathbf{u} = \nabla f/\lVert\nabla f\rVert$.
(a) The maximal slope is therefore $\lVert\nabla f\rVert$, and the most negative slope,
$-\lVert\nabla f\rVert$, is attained along $-\nabla f$ — the direction gradient descent takes.
(b) The slope is zero when $\cos\phi = 0$, i.e. for every $\mathbf{u}$ orthogonal to the gradient.
(c) Moving along the level set $\{f = c\}$ leaves $f$ unchanged, so every tangent direction of
the level set has zero directional derivative and is orthogonal to $\nabla f$: the gradient is
perpendicular to the level set. The colleague's component-wise statement misses that the
*ratio* of the components matters — walking along $(1, 0)$ when $\nabla f = (1, 100)$ is uphill
but almost flat compared with walking along $(1, 100)/\sqrt{10001}$.

</details>

## Exercise 5 — Analytic gradient of a two-variable function  `★★☆`

Let $f(x, y) = x^2 y + \sin(xy) + e^{y}$. The function is given as `f4(v)` below (with `v = (x, y)`).
Work out $\partial f/\partial x$ and $\partial f/\partial y$ by hand (product rule for $x^2 y$,
chain rule for $\sin(xy)$) and implement `grad_f4(v)` returning the gradient as a length-2 array.

The checks compare your gradient at $(1, 2)$ and at $(1.5, -1)$ against hidden values, so a
finite-difference shortcut would pass — but write the analytic formula; that is the skill.

In [ ]:
def f4(v):
    x, y = v
    return x**2 * y + np.sin(x * y) + np.exp(y)

def grad_f4(v):
    x, y = v
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    got_a = grad_f4(np.array([1.0, 2.0]))
    got_b = grad_f4(np.array([1.5, -1.0]))
except Exception:
    got_a, got_b = None, None
check('gradient has shape (2,)', got_a is not None and np.shape(got_a) == (2,))
check_hash('hidden: gradient at (1, 2)', got_a, 'fe946181ca211e3d')
check_hash('hidden: gradient at (1.5, -1)', got_b, 'b31e1f801fe2d30d')
try:
    got_c = grad_f4(np.array([-0.4, 0.9]))
except Exception:
    got_c = None
check_close('matches a finite-difference gradient at (-0.4, 0.9)', got_c, reference_grad_fd(f4, np.array([-0.4, 0.9])), tol=1e-6)

## Exercise 6 — Jacobian of a map from R^2 to R^3  `★★☆`

The map $\mathbf{F}(x, y) = \bigl(xy,\; x^2 + y^2,\; e^{x}\sin y\bigr)$ is given as `F5(v)`.
Implement `jac_F5(v)` returning its Jacobian as a $3\times 2$ array with
$J_{ij} = \partial F_i/\partial x_j$ (rows = outputs, columns = inputs).

Then convince yourself of the linearisation $\mathbf{F}(\mathbf{v}+\boldsymbol{\delta}) \approx \mathbf{F}(\mathbf{v}) + J\boldsymbol{\delta}$:
the last check verifies that the error is tiny for a small $\boldsymbol{\delta}$.

In [ ]:
def F5(v):
    x, y = v
    return np.array([x * y, x**2 + y**2, np.exp(x) * np.sin(y)])

def jac_F5(v):
    x, y = v
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    J = np.asarray(jac_F5(np.array([0.5, 1.0])), dtype=float)
except Exception:
    J = None
check('Jacobian has shape (3, 2): 3 outputs, 2 inputs', J is not None and J.shape == (3, 2))
check_hash('hidden: Jacobian at (0.5, 1.0)', J, 'd7808ca6832d576c')
try:
    J0 = jac_F5(np.array([0.0, 0.0]))
except Exception:
    J0 = None
check_close('Jacobian at the origin is [[0,0],[0,0],[0,1]]', J0, np.array([[0., 0.], [0., 0.], [0., 1.]]))
try:
    v0, d = np.array([0.5, 1.0]), np.array([1e-3, -2e-3])
    lin_err = np.linalg.norm(F5(v0 + d) - F5(v0) - jac_F5(v0) @ d)
except Exception:
    lin_err = None
check('linearisation error for |delta| ~ 2e-3 is below 1e-5 (quadratic in delta)', lin_err is not None and lin_err < 1e-5)

## Exercise 7 — Hessian and classification of critical points  `★★☆`

Let $g(x, y) = x^4 + y^4 - 4xy$ (given as `g6`, with gradient `grad_g6` provided). Its critical
points are $(0,0)$, $(1,1)$ and $(-1,-1)$ — verify this from $\nabla g = \mathbf{0}$.

1. Implement `hess_g6(v)` returning the $2\times2$ Hessian (an array).
2. Implement `classify(H, tol=1e-8)` that returns one of the strings `"minimum"`, `"maximum"`,
   `"saddle"` or `"degenerate"` from the eigenvalues of a symmetric matrix `H`
   (use `np.linalg.eigvalsh`; treat eigenvalues with $|\lambda| \le$ `tol` as zero).

The hidden checks look at the eigenvalues at $(1, 1)$ and the classification of the two kinds of
critical point.

In [ ]:
def g6(v):
    x, y = v
    return x**4 + y**4 - 4 * x * y

def grad_g6(v):
    x, y = v
    return np.array([4 * x**3 - 4 * y, 4 * y**3 - 4 * x])

def hess_g6(v):
    x, y = v
    # YOUR CODE HERE
    return None

def classify(H, tol=1e-8):
    # YOUR CODE HERE: return "minimum", "maximum", "saddle" or "degenerate"
    return None

**Check your answer:**

In [ ]:
try:
    H11 = np.asarray(hess_g6(np.array([1.0, 1.0])), dtype=float)
    lam11 = np.linalg.eigvalsh(H11)
except Exception:
    H11, lam11 = None, None
check('Hessian has shape (2, 2) and is symmetric', H11 is not None and H11.shape == (2, 2) and np.allclose(H11, H11.T))
check_hash('hidden: eigenvalues of the Hessian at (1, 1)', lam11, 'b0748f52d2217e5e')
try:
    c0 = classify(hess_g6(np.array([0.0, 0.0])))
    c1 = classify(hess_g6(np.array([1.0, 1.0])))
    cm = classify(hess_g6(np.array([-1.0, -1.0])))
except Exception:
    c0, c1, cm = None, None, None
check_hash('hidden: classification of the origin', c0, 'ba08dad18ac84822')
check_hash('hidden: classification of (1, 1)', c1, '692e4e5decffcf5e')
check('(1, 1) and (-1, -1) are classified the same way', c1 is not None and c1 == cm)
try:
    got = classify(np.array([[-2.0, 0.0], [0.0, -1.0]]))
except Exception:
    got = None
check('negative definite Hessian is a maximum', got == 'maximum')
try:
    got = classify(np.array([[1.0, 0.0], [0.0, 0.0]]))
except Exception:
    got = None
check('a zero eigenvalue is degenerate', got == 'degenerate')

## Exercise 8 — Taylor polynomials from derivative values  `★★☆`

1. Implement `taylor_coeffs(derivs)`: given the list $[f(a), f'(a), f''(a), \dots, f^{(k)}(a)]$ of
   derivative values at a point $a$, return the array of Taylor coefficients
   $c_j = f^{(j)}(a)/j!$ for $j = 0,\dots,k$.
2. Implement `taylor_eval(coeffs, a, x)` returning $T_k(x) = \sum_j c_j (x-a)^j$ (vectorised over
   a NumPy array `x`).

For $\ln(1+x)$ about $a = 0$ the derivatives are $0, 1, -1, 2, -6, \dots$ (that is,
$f^{(j)}(0) = (-1)^{j+1}(j-1)!$ for $j\ge1$). For $e^x$ about $0$ they are all $1$.

In [ ]:
from math import factorial

def taylor_coeffs(derivs):
    # YOUR CODE HERE
    return None

def taylor_eval(coeffs, a, x):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    c = np.asarray(taylor_coeffs([0.0, 1.0, -1.0, 2.0, -6.0]), dtype=float)
except Exception:
    c = None
check_hash('hidden: degree-4 Taylor coefficients of ln(1+x) about 0', c, 'a5e3e18de34f00de')
try:
    got = taylor_eval(taylor_coeffs([1.0] * 4), 0.0, 0.5)
except Exception:
    got = None
check_close('degree-3 Taylor polynomial of exp at 0.5 is 1.645833...', got, 1 + 0.5 + 0.125 + 0.125 / 6, tol=1e-9)
try:
    xs = np.array([0.05, 0.1, 0.2])
    got = taylor_eval(c, 0.0, xs)
    err = np.abs(got - np.log1p(xs))
except Exception:
    got, err = None, None
check('taylor_eval is vectorised over x', got is not None and np.shape(got) == (3,))
check('degree-4 model of ln(1+x) is within 1e-4 at x = 0.2 but the error grows with x',
      err is not None and err[2] < 1e-4 and err[0] < err[1] < err[2])
try:
    got = taylor_eval(taylor_coeffs([np.e, np.e, np.e]), 1.0, 1.1)
except Exception:
    got = None
check_close('expansion of exp about a = 1 evaluated at 1.1', got, np.e * (1 + 0.1 + 0.005), tol=1e-9)

## Exercise 9 — Gradient of least squares and the normal equations  `★★☆`

For $L(\mathbf{x}) = \lVert A\mathbf{x} - \mathbf{b}\rVert^2$ implement

- `ls_loss(x, A, b)` returning the scalar loss,
- `ls_grad(x, A, b)` returning the analytic gradient $2A^\top(A\mathbf{x}-\mathbf{b})$ (derive it
  from $\nabla \mathbf{x}^\top M\mathbf{x} = 2M\mathbf{x}$ and $\nabla \mathbf{c}^\top\mathbf{x} = \mathbf{c}$,
  or via the chain rule with the Jacobian $A$ of the residual), and
- `normal_eq_solve(A, b)` that returns the minimiser by solving $A^\top A\mathbf{x} = A^\top\mathbf{b}$
  with `np.linalg.solve` — this is "set the gradient to zero".

The checks use `A = rng.standard_normal((8, 3))`, `b = rng.standard_normal(8)` drawn from a
fresh `np.random.default_rng(0)` and the point `x0 = (0.5, -1, 2)`.

In [ ]:
def ls_loss(x, A, b):
    # YOUR CODE HERE
    return None

def ls_grad(x, A, b):
    # YOUR CODE HERE
    return None

def normal_eq_solve(A, b):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
_r = np.random.default_rng(0)
A, b, x0 = _r.standard_normal((8, 3)), _r.standard_normal(8), np.array([0.5, -1.0, 2.0])
try:
    got = ls_loss(x0, A, b)
except Exception:
    got = None
check_close('loss at x0 equals the squared residual norm', got, float(np.sum((A @ x0 - b)**2)), tol=1e-9)
try:
    g = np.asarray(ls_grad(x0, A, b), dtype=float)
except Exception:
    g = None
check('gradient has shape (3,)', g is not None and g.shape == (3,))
check_hash('hidden: gradient at x0', g, '6a17bb847f299c27')
try:
    g_ref = reference_grad_fd(lambda v: ls_loss(v, A, b), x0)
except Exception:
    g_ref = None
check_close('gradient matches finite differences', g, g_ref, tol=1e-5)
try:
    xs = np.asarray(normal_eq_solve(A, b), dtype=float)
    g_star = np.asarray(ls_grad(xs, A, b), dtype=float)
except Exception:
    xs, g_star = None, None
check_hash('hidden: least-squares solution', xs, 'f34e803b618009dc')
check('gradient vanishes at the normal-equation solution', g_star is not None and np.linalg.norm(g_star) < 1e-9)
check('normal-equation solution agrees with np.linalg.lstsq', xs is not None and np.allclose(xs, np.linalg.lstsq(A, b, rcond=None)[0]))

## Exercise 10 — Gradient of the logistic loss  `★★☆`

For logistic regression with $p_i = \sigma(\mathbf{w}^\top\mathbf{x}_i)$ and the mean binary
cross-entropy $L(\mathbf{w}) = -\frac1n\sum_i\bigl[y_i\ln p_i + (1-y_i)\ln(1-p_i)\bigr]$, implement

- `bce_loss(w, X, y)` (a float; `X` is $n\times d$, `y` is a 0/1 array of length $n$), and
- `bce_grad(w, X, y)` returning $\nabla_{\mathbf{w}}L = \frac1n X^\top(\mathbf{p}-\mathbf{y})$ — derive
  $\partial L_i/\partial z_i = p_i - y_i$ first (Section 8 of the guide).

Tiny dataset for the checks:
```python
X = np.array([[1.0, 0.5, -1.0], [1.0, -1.5, 0.5], [1.0, 2.0, 1.0], [1.0, 0.0, -0.5]])
y = np.array([1.0, 0.0, 1.0, 0.0]);   w = np.array([0.1, -0.2, 0.3])
```

In [ ]:
def bce_loss(w, X, y):
    # YOUR CODE HERE
    return None

def bce_grad(w, X, y):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
X = np.array([[1.0, 0.5, -1.0], [1.0, -1.5, 0.5], [1.0, 2.0, 1.0], [1.0, 0.0, -0.5]])
y = np.array([1.0, 0.0, 1.0, 0.0]); w = np.array([0.1, -0.2, 0.3])
try:
    L = float(bce_loss(w, X, y))
except Exception:
    L = None
check_hash('hidden: loss at w', L, 'f339286c3d27672e')
try:
    L0 = float(bce_loss(np.zeros(3), X, y))
except Exception:
    L0 = None
check_close('loss at w = 0 is ln 2', L0, np.log(2.0), tol=1e-9)
try:
    g = np.asarray(bce_grad(w, X, y), dtype=float)
except Exception:
    g = None
check('gradient has shape (3,)', g is not None and g.shape == (3,))
check_hash('hidden: gradient at w', g, '8185832ad1452d58')
try:
    g_ref = reference_grad_fd(lambda v: bce_loss(v, X, y), w)
except Exception:
    g_ref = None
check_close('gradient matches finite differences', g, g_ref, tol=1e-6)
try:
    L_after = float(bce_loss(w - 0.5 * g, X, y))
except Exception:
    L_after = None
check('one gradient step (eta = 0.5) lowers the loss', L is not None and L_after is not None and L_after < L)

## Exercise 11 — Hessian-vector products and the sharpest direction  `★★★`

For a billion parameters the Hessian cannot be stored, but the *product* $H\mathbf{v}$ can be
computed from gradients alone, because $H$ is the Jacobian of $\nabla f$:

$$H\mathbf{v} \approx \frac{\nabla f(\mathbf{x} + h\mathbf{v}) - \nabla f(\mathbf{x} - h\mathbf{v})}{2h}.$$

1. Implement `hvp_fd(grad_f, x, v, h=1e-5)` using this formula (`grad_f` returns the gradient
   array at a point).
2. Implement `top_curvature(grad_f, x, n_iter=200, seed=0)` that runs **power iteration** with
   `hvp_fd` to estimate the largest eigenvalue $\lambda_{\max}$ of the Hessian at `x` (start from a
   random unit vector drawn with `np.random.default_rng(seed)`, repeatedly set
   $\mathbf{v} \leftarrow H\mathbf{v}/\lVert H\mathbf{v}\rVert$, and return the tuple
   `(lam, v)` with `lam = v @ (H v)` as a float). Assume the Hessian's largest-magnitude eigenvalue
   is positive, as it is for the test functions.

Test functions: the quadratic $\tfrac12\mathbf{x}^\top H\mathbf{x}$ with
$H = \begin{pmatrix}2&3\\3&10\end{pmatrix}$ (gradient $H\mathbf{x}$, eigenvalues $1$ and $11$) and
$f(\mathbf{x}) = \sum_i x_i^4 + \mathbf{x}^\top M\mathbf{x}$ with gradient $4\mathbf{x}^3 + 2M\mathbf{x}$
for the symmetric $M$ given in the check cell, at $\mathbf{x} = (0.5, -1, 0.25)$. Recall from the
lesson that $2/\lambda_{\max}$ is the largest stable learning rate for gradient descent.

In [ ]:
def hvp_fd(grad_f, x, v, h=1e-5):
    # YOUR CODE HERE
    return None

def top_curvature(grad_f, x, n_iter=200, seed=0):
    # YOUR CODE HERE: power iteration using hvp_fd; return (lam, v)
    return None

**Check your answer:**

In [ ]:
H = np.array([[2.0, 3.0], [3.0, 10.0]])
grad_quad = lambda x: H @ x
try:
    got = hvp_fd(grad_quad, np.array([1.0, -1.0]), np.array([1.0, 2.0]))
except Exception:
    got = None
check_close('H v for the quadratic equals the matrix-vector product (8, 23)', got, H @ np.array([1.0, 2.0]), tol=1e-6)
try:
    lam, v = top_curvature(grad_quad, np.zeros(2))
    lam = float(lam)
except Exception:
    lam, v = None, None
check_close('largest curvature of the quadratic is 11', lam, 11.0, tol=1e-5)
check('returned eigenvector is a unit vector aligned with (0.316, 0.949)',
      v is not None and abs(np.linalg.norm(v) - 1) < 1e-6 and abs(abs(np.dot(v, [0.31622777, 0.9486833])) - 1) < 1e-4)

M = np.array([[3.0, 1.0, 0.0], [1.0, 2.0, 0.5], [0.0, 0.5, 1.0]])
grad_quartic = lambda x: 4 * x**3 + 2 * M @ x
x0 = np.array([0.5, -1.0, 0.25])
try:
    lam2 = float(top_curvature(grad_quartic, x0)[0])
except Exception:
    lam2 = None
check_hash('hidden: largest curvature of the quartic at x0 (2 decimals)', lam2, '410b73656f27aca0', decimals=2)
check('stability limit 2/lambda_max for the quartic lies between 0.11 and 0.13', lam2 is not None and 0.11 < 2 / lam2 < 0.13)

## Exercise 12 — Mini-project: a reusable gradient checker  `★★★`

Analytic gradients are where most bugs in hand-written ML code live. Build the tool that catches
them and use it to debug a real (small) model.

1. Write `grad_check(f, grad_f, x, h=1e-6)` for any scalar function `f` of a 1-D parameter vector
   `x` and a candidate gradient function `grad_f`. It should return the **relative error**

   $$\frac{\lVert \mathbf{g}_{\text{analytic}} - \mathbf{g}_{\text{numeric}}\rVert}{\lVert \mathbf{g}_{\text{analytic}}\rVert + \lVert \mathbf{g}_{\text{numeric}}\rVert}$$

   together with both gradients, using central differences for $\mathbf{g}_{\text{numeric}}$.
2. The starter cell defines a two-layer *linear* model $\hat{\mathbf{y}} = XW_1W_2$ with MSE loss
   $L = \frac1n\lVert XW_1W_2 - \mathbf{y}\rVert^2$, and a colleague's gradient `grad_loss_buggy`.
   Run your checker on it. It should fail.
3. Derive the correct gradient with the chain rule (hint: with $\mathbf{r} = XW_1W_2 - \mathbf{y}$ and
   $\mathbf{h} = XW_1$, $\partial L/\partial\mathbf{r} = 2\mathbf{r}/n$; then
   $\nabla_{W_2}L = H^\top(\partial L/\partial\mathbf{r})$ and $\nabla_{W_1}L = X^\top(\partial L/\partial\mathbf{r})W_2^\top$).
   Use the *ratio* of numerical to analytic entries to locate the bug, write `grad_loss`, and show
   the checker passes (relative error below $10^{-6}$).
4. Plot the relative error of the correct gradient against $h$ from $10^{-12}$ to $10^{-1}$. You
   will *not* see the V shape from the lesson — explain why (what is the third derivative of the
   loss along a single coordinate of $W_1$ or $W_2$?). Then replace the hidden layer by
   $\tanh(XW_1)$, derive the gradient of the new model (one extra factor $1 - \tanh^2$ in the
   chain), check it, and plot its error curve: the truncation arm should reappear.
5. Take a few gradient steps with your corrected gradient and confirm the loss decreases.

**Rubric — you have mastered this when you can:**

- `grad_check` works for any `f`/`grad_f` pair and any dimension, never modifies `x` in place, and
  returns a relative error that is scale-free (dividing by the sum of the norms, with a guard
  against division by zero).
- The buggy gradient is flagged with a relative error far above $10^{-6}$, and you can say *which
  constants* are missing from each block (a factor $2$ in $W_2$, a factor $2/n$ in $W_1$).
- The corrected gradient is derived with the chain rule, written in matrix form with the right
  shapes ($d\times k$ and $k$), and passes with relative error around $10^{-9}$ or better.
- You can explain why the linear model's error curve has no truncation arm (the loss is exactly
  quadratic along each coordinate, so the central difference is exact and only round-off remains)
  and why the tanh model's curve shows the V shape with a minimum near $h \approx 10^{-5}$.
- The tanh model's gradient is derived with one more chain-rule factor, $1 - \tanh^2$, applied
  elementwise to the hidden pre-activations, and passes the checker.
- A few gradient-descent steps with the corrected gradient reduce the loss.

In [ ]:
# --- data and a two-layer *linear* model:  y_hat = X @ W1 @ W2,  loss = mean((y_hat - y)^2) ---
rng_p = np.random.default_rng(1)
n, d, k = 50, 4, 3
X = rng_p.standard_normal((n, d))
y = X @ rng_p.standard_normal(d) + 0.1 * rng_p.standard_normal(n)
W1 = 0.5 * rng_p.standard_normal((d, k))
W2 = 0.5 * rng_p.standard_normal(k)

def pack(W1, W2):
    return np.concatenate([W1.ravel(), W2.ravel()])

def unpack(theta):
    return theta[:d * k].reshape(d, k), theta[d * k:]

def loss(theta):
    W1, W2 = unpack(theta)
    r = X @ W1 @ W2 - y
    return float(np.mean(r**2))

def grad_loss_buggy(theta):
    # A colleague's hand-derived gradient. It contains at least one mistake. Find it with grad_check.
    W1, W2 = unpack(theta)
    hdn = X @ W1                      # n x k
    r = hdn @ W2 - y                  # n
    gW2 = hdn.T @ r / n               # dL/dW2
    gW1 = X.T @ np.outer(r, W2)       # dL/dW1
    return pack(gW1, gW2)

def grad_check(f, grad_f, x, h=1e-6):
    # YOUR CODE HERE: return (relative_error, numerical_gradient, analytic_gradient)
    return None

# Your mini-project workspace: write grad_check, run it on grad_loss_buggy, fix the gradient,
# then plot the relative error versus h to see the round-off regime.

## Score

In [ ]:
summary()